# Deutschlandticket Analysis — Full Pipeline

This notebook runs the six-step analysis pipeline end-to-end.
For the final presentation with charts and conclusions, see **`final_summary.ipynb`**.

In [ ]:
import sys
from pathlib import Path

ROOT = Path('..').resolve()
sys.path.insert(0, str(ROOT))

import pandas as pd
from src.synthetic_data import main as run_synthetic_data
from src.pt_connection import run_pt_connection
from src.commute import run_commute
from src.grouping import run_grouping
from src.scoring import run_scoring
from src.summary import run_summary
from src.map_visualization import generate_map, load_stations

## Run pipeline (skip steps whose outputs already exist)

In [ ]:
data_path = ROOT / 'data' / 'employees_synthetic.csv'

if not data_path.exists():
    run_synthetic_data()

employees = pd.read_csv(data_path)
if employees['pt_access_score'].isna().any():
    run_pt_connection()
if 'transit_time_minutes' not in employees.columns or employees['transit_time_minutes'].isna().any():
    run_commute()
if 'commute_group' not in employees.columns:
    run_grouping()
if 'adoption_score' not in employees.columns:
    run_scoring()

results = run_summary()
employees = pd.read_csv(data_path)
print(f'Pipeline complete — {len(employees)} employees')
employees.head()

In [ ]:
from src.scoring import organization_summary

org = organization_summary(employees)
print('=== Organization Summary ===')
for key, value in org.items():
    print(f'{key}: {value}')

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
employees['adoption_score'].hist(ax=axes[0], bins=20, edgecolor='black')
axes[0].set_title('Adoption Score Distribution')
axes[0].set_xlabel('adoption_score')

employees['ticket_recommendation'].value_counts().plot.bar(ax=axes[1], color=['#2ecc71', '#f39c12', '#e74c3c'])
axes[1].set_title('Ticket Recommendations')
plt.tight_layout()
plt.show()

In [ ]:
stations = load_stations(ROOT / 'data' / 'stations.csv')
map_path = generate_map(employees, stations, ROOT / 'output' / 'employee_map.html')
print(f'Map saved to {map_path}')